## Agentic Workflow with LangGraph

In [7]:
import os
from typing import Annotated
import operator
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage
from langgraph.graph import MessagesState
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver
from groq import Groq


In [8]:
class TicketState(TypedDict):
    customer_message: str
    log: Annotated[list, operator.add]
 
def log_received(state: TicketState) -> dict:
    return {"log": [f"Received: {state['customer_message']}"]}
 
def log_assigned(state: TicketState) -> dict:
    return {"log": ["Assigned to support queue"]}


In [9]:
builder = StateGraph(TicketState)
builder.add_node("log_received", log_received)
builder.add_node("log_assigned", log_assigned)
builder.add_edge(START, "log_received")
builder.add_edge("log_received", "log_assigned")
builder.add_edge("log_assigned", END)
graph = builder.compile()
 
result = graph.invoke({"customer_message": "My invoice looks wrong", "log": []})
print(result)

{'customer_message': 'My invoice looks wrong', 'log': ['Received: My invoice looks wrong', 'Assigned to support queue']}


Run this in your terminal

export GROQ_API_KEY="your-key-here"
or 
export OPENAI_API_KEY="your-key-here"

In [12]:
from dotenv import load_dotenv
load_dotenv()  # reads .env and sets os.environ

groq_key = os.getenv("GROQ_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

In [21]:
class LLMClient:
    """
    Unified wrapper around Groq and OpenAI chat models.
    Both backends expose the same LangChain `.invoke(messages)` interface,
    so callers never need to know which provider is active.
    """

    def __init__(
        self,
        groq_model: str = "openai/gpt-oss-20b",
        openai_model: str = "gpt-4o-mini",
        temperature: float = 0.0,
    ):
        self.groq_api_key = groq_key if groq_key != "" else None
        self.openai_api_key = openai_key if openai_key != "" else None
        self.groq_model = groq_model
        self.openai_model = openai_model
        self.temperature = temperature
        self.provider, self.llm = self._init_llm()

    def _init_llm(self):
        if self.groq_api_key:
            return "groq", ChatGroq(
                api_key=self.groq_api_key,
                model=self.groq_model,
                temperature=self.temperature,
            )
        if self.openai_api_key:
            return "openai", ChatOpenAI(
                model=self.openai_model,
                temperature=self.temperature,
            )
        raise ValueError("No GROQ_API_KEY or OPENAI_API_KEY found in environment.")

    def invoke(self, messages):
        """Delegate to the underlying LangChain chat model."""
        return self.llm.invoke(messages)

    def __repr__(self):
        return f"<LLMClient provider={self.provider} model={self.groq_model if self.provider=='groq' else self.openai_model}>"


# Instantiate once, reuse everywhere
llm_client = LLMClient()


llm = llm_client.llm  # unwrap the underlying LangChain model
tools = [get_customer_tier]
llm_with_tools = llm.bind_tools(tools)

def run_model(state: MessagesState) -> dict:
    system = SystemMessage("You are a support agent for a SaaS product. "
                           "Use available tools when you need account-specific information.")
    response = llm_with_tools.invoke([system] + state["messages"])
    return {"messages": [response]}

In [22]:
builder = StateGraph(MessagesState)
builder.add_node("run_model", run_model)
builder.add_edge(START, "run_model")
builder.add_edge("run_model", END)
 
graph = builder.compile()
 
result = graph.invoke({"messages": [HumanMessage("My dashboard isn't loading. What should I try?")]})
print(result["messages"][-1].content)

In [23]:
@tool
def get_customer_tier(customer_id: str) -> str:
    """Look up the subscription tier for a customer by their ID.
    Returns 'free', 'pro', or 'enterprise'."""
    tiers = {
        "cust_1001": "enterprise",
        "cust_2002": "pro",
        "cust_3003": "free",
    }
    return tiers.get(customer_id, "not found")

In [24]:
tools = [get_customer_tier]
llm_with_tools = llm.bind_tools(tools)
 
def run_model(state: MessagesState) -> dict:
    system = SystemMessage("You are a support agent for a SaaS product. "
                           "Use available tools when you need account-specific information.")
    response = llm_with_tools.invoke([system] + state["messages"])
    return {"messages": [response]}

In [25]:
tool_node = ToolNode(tools)
 
builder = StateGraph(MessagesState)
builder.add_node("run_model", run_model)
builder.add_node("tools", tool_node)
 
builder.add_edge(START, "run_model")
builder.add_conditional_edges("run_model", tools_condition)
builder.add_edge("tools", "run_model")
 
graph = builder.compile()

In [26]:
result = graph.invoke({"messages": [
    HumanMessage("Can you check what plan customer cust_1001 is on?")
]})
 
for msg in result["messages"]:
    print(type(msg).__name__, ":", msg.content or msg.tool_calls)

HumanMessage : Can you check what plan customer cust_1001 is on?
AIMessage : [{'name': 'get_customer_tier', 'args': {'customer_id': 'cust_1001'}, 'id': 'fc_6fc4d336-59a9-4de4-b4c8-87d983792ad3', 'type': 'tool_call'}]
ToolMessage : enterprise
AIMessage : Customer **cust_1001** is on the **Enterprise** plan.


In [27]:
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [28]:
config = {"configurable": {"thread_id": "ticket-7741"}}
 
graph.invoke(
    {"messages": [HumanMessage("Hi, I can't access my account.")]},
    config,
)
 
result = graph.invoke(
    {"messages": [HumanMessage("My ID is cust_2002, can you check my plan?")]},
    config,
)
 
print(result["messages"][-1].content)

Thanks for providing your ID. I’ve checked your account and you’re on the **Pro** plan. Let’s get you back into your account.

**Here are a few quick steps to try:**

1. **Reset your password**  
   - Click the “Forgot password?” link on the login page.  
   - Enter the email associated with `cust_2002` and follow the instructions in the email that’s sent.

2. **Clear browser cache / try a different browser**  
   - Sometimes cached data can cause login issues.  
   - If you’re on Chrome, try incognito mode or switch to Firefox/Edge.

3. **Check for service status**  
   - We’re currently running smoothly, but you can view our status page here: [https://status.yourapp.com](https://status.yourapp.com) to confirm there are no outages.

4. **Two‑factor authentication (if enabled)**  
   - If you have 2FA set up, make sure you’re using the correct authenticator app or SMS code.

If none of these resolve the issue, let me know the exact error message you’re seeing (or a screenshot if you ca

Checkpointers persist graph state for a thread. If your application also needs to persist data independently of any conversation, such as user profiles, preferences, or long-term memories shared across multiple threads, use a Store. Stores complement checkpointers by providing durable application-level storage that graphs can access during execution.